In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU Memory: {total_memory:.1f} GB")
    
    # Clear cache
    torch.cuda.empty_cache()

CUDA available: True
GPU: NVIDIA GeForce RTX 2060
Total GPU Memory: 6.4 GB

In [7]:
# install unsloth first
# pip install unsloth
import sys
import types
from transformers import modeling_utils

# Monkey patch for shard_checkpoint
def shard_checkpoint(state_dict, max_shard_size, weights_name):
    """
    A simple shard_checkpoint implementation that doesn't actually shard.
    This is a fallback for older transformers versions.
    """
    # Just return the original state dict without sharding
    return [(state_dict, weights_name)]

# Attach the function to modeling_utils if it doesn't exist
if not hasattr(modeling_utils, 'shard_checkpoint'):
    modeling_utils.shard_checkpoint = shard_checkpoint
    print("✓ Monkey patched shard_checkpoint into modeling_utils")

# Also add to the module if needed for direct imports
if not hasattr(modeling_utils, 'shard_checkpoint'):
    setattr(modeling_utils, 'shard_checkpoint', shard_checkpoint)

# Now import unsloth
from unsloth import FastLanguageModel
import torch


from unsloth import FastLanguageModel
import torch
from transformers import BitsAndBytesConfig

# Simplified 4-bit quantization config (no CPU offloading)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=2048,
    load_in_4bit=True,
    quantization_config=bnb_config,
)

# Continue with LoRA and training...

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
)

from datasets import Dataset

# Convert your list to a Hugging Face Dataset
training_data = Dataset.from_list([
    {"prompt": "What's the weather in Paris?", 
     "completion": "I'll use the get_weather tool for Paris."},
    # ... more examples
])

# Fine-tune
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = training_data,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs"
    ),
)
trainer.train()

==((====))==  Unsloth 2026.3.10: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 2060. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

Unsloth: Tokenizing ["prompt"+"completion"]:   0%|          | 0/1 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)

Step,Training Loss


TrainOutput(global_step=100, training_loss=0.5504482073719918, metrics={'train_runtime': 45.6092, 'train_samples_per_second': 17.54, 'train_steps_per_second': 2.193, 'total_flos': 14198746521600.0, 'train_loss': 0.5504482073719918, 'epoch': 100.0})